# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

# Data Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Making the column 'CID' useful to connect with other tables

In [0]:
df = (
    df
    .withColumn(
        'CID',
        F.regexp_replace(col("CID"), "-", "")
    )
)

### Country Normalization

In [0]:
df = df.withColumn(
    'cntry',
    F.when(F.upper(col('cntry')).isin('US', 'USA', 'UNITED STATES'), 'United States')
     .when(F.upper(col('cntry')).isin('DE', 'GERMANY'), 'Germany')
     .when((col('cntry') == '') | (col('cntry').isNull()), 'Unknown')
     .otherwise(col('cntry'))
)

# Renaming the columns

In [0]:
renaming_map = {
    'cid': 'customer_number',
    'cntry': 'country'
}

for old_name, new_name in renaming_map.items():
    df = df.withColumnRenamed(old_name, new_name)

# Checking DataFrame

In [0]:
df.display()

# Writing into Silver Table

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.erp_customer_location')
)